# Reciprocal Rank Fusion [Step 3 - Merging BM25 and Dense Results]

> **MLCourse - Agentic AI - Hybrid Search**

This notebook explains Reciprocal Rank Fusion (RRF), the algorithm
used to combine ranked lists from different retrieval systems into a
single merged ranking. We build both a BM25 index and a dense vector
index, then fuse their results with RRF.

In [1]:
# Import all libraries needed for this notebook.
import re                              # Tokenization
import math                            # RRF math
from rank_bm25 import BM25Okapi        # BM25 keyword search
import chromadb                        # Dense vector store
from chromadb.utils import embedding_functions  # Embedding models

In [2]:
# ## Part 1: Building Both Indexes
# To fuse results, we first need results from both retrieval systems.
# We load Alice in Wonderland and build a BM25 index and a ChromaDB
# collection in parallel.

CORPUS_PATH = r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt"

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Split into paragraphs and filter short ones.
paragraphs = [p.strip() for p in raw_text.split("\n\n") if len(p.strip()) > 80]

# Tokenize for BM25.
def simple_tokenize(text):
    """Lowercase and split on non-alphanumeric characters."""
    return re.findall(r"[a-z0-9]+", text.lower())

tokenized_corpus = [simple_tokenize(doc) for doc in paragraphs]

# Build BM25 index.
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 index: {len(paragraphs)} documents")

# Build ChromaDB dense index.
CHROMA_DIR = r"D:\projects\python\MLCourse\03_agentic_ai\20_hybrid_search\chroma_rrf"

import shutil, os
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

client = chromadb.PersistentClient(path=CHROMA_DIR)
ef = embedding_functions.DefaultEmbeddingFunction()
dense_collection = client.get_or_create_collection(
    name="alice_dense",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)

# Index paragraphs in batches.
BATCH_SIZE = 100
for i in range(0, len(paragraphs), BATCH_SIZE):
    batch = paragraphs[i : i + BATCH_SIZE]
    ids = [f"doc_{j}" for j in range(i, i + len(batch))]
    dense_collection.add(documents=batch, ids=ids)

print(f"Dense index: {dense_collection.count()} documents")

BM25 index: 546 documents


Dense index: 546 documents


In [3]:
# ## Part 2: Getting Separate Rankings
# We run the same query through both systems and collect their ranked lists.

query = "the queen and the croquet game"
print(f"Query: '{query}'")
print()

# BM25 ranking.
query_tokens = simple_tokenize(query)
bm25_scores = bm25.get_scores(query_tokens)
bm25_ranked = bm25_scores.argsort()[::-1]  # Indices sorted by score desc

# Dense ranking.
dense_results = dense_collection.query(
    query_texts=[query], n_results=len(paragraphs)
)
# ChromaDB returns IDs; map them back to document indices.
dense_ids = dense_results["ids"][0]
dense_ranked = [int(d.split("_")[1]) for d in dense_ids]

print("BM25 top 10:")
for rank, idx in enumerate(bm25_ranked[:10], 1):
    preview = paragraphs[idx][:60].replace("\n", " ")
    print(f"  Rank {rank}: doc_{idx} -- {preview}...")

print()
print("Dense top 10:")
for rank, idx in enumerate(dense_ranked[:10], 1):
    preview = paragraphs[idx][:60].replace("\n", " ")
    print(f"  Rank {rank}: doc_{idx} -- {preview}...")

Query: 'the queen and the croquet game'

BM25 top 10:
  Rank 1: doc_365 -- “Let’s go on with the game,” the Queen said to Alice; and Al...
  Rank 2: doc_173 -- The Fish-Footman began by producing from under his arm a gre...
  Rank 3: doc_315 -- “Get to your places!” shouted the Queen in a voice of thunde...
  Rank 4: doc_213 -- “Call it what you like,” said the Cat. “Do you play croquet ...
  Rank 5: doc_366 -- The other guests had taken advantage of the Queen’s absence,...
  Rank 6: doc_199 -- “Here! you may nurse it a bit, if you like!” the Duchess sai...
  Rank 7: doc_330 -- Alice thought she might as well go back, and see how the gam...
  Rank 8: doc_0 -- CHAPTER I.     Down the Rabbit-Hole  CHAPTER II.    The Pool...
  Rank 9: doc_303 -- “Get up!” said the Queen, in a shrill, loud voice, and the t...
  Rank 10: doc_340 -- The Cat’s head began fading away the moment he was gone, and...

Dense top 10:
  Rank 1: doc_213 -- “Call it what you like,” said the Cat. “Do you play croquet .

In [4]:
# ## Part 3: Reciprocal Rank Fusion (RRF) Explained
# RRF merges multiple ranked lists into one combined ranking.
#
# Formula for each document d:
#   RRF_score(d) = SUM over all ranked lists i of [ 1 / (k + rank_i(d)) ]
#
# where:
#   rank_i(d) = position of document d in ranked list i (1-indexed)
#   k = smoothing constant (typically 60)
#
# The constant k prevents top-ranked documents from dominating.
# With k=60: rank 1 contributes 1/61 = 0.0164
#             rank 2 contributes 1/62 = 0.0161
#             rank 10 contributes 1/70 = 0.0143
#
# The differences are small, so documents appearing in multiple
# lists get a natural boost.

K = 60  # Standard RRF smoothing constant

def rrf_score(doc_index, ranked_lists, k=60):
    """Compute RRF score for a document across multiple ranked lists."""
    score = 0.0
    for ranked in ranked_lists:
        rank = list(ranked).index(doc_index) + 1  # 1-indexed rank
        score += 1.0 / (k + rank)
    return score

print("RRF formula: score(d) = SUM_i [ 1 / (k + rank_i(d)) ]")
print(f"Smoothing constant k = {K}")
print()

# Show how individual contributions work.
print("Example contribution values (k=60):")
for rank in [1, 2, 5, 10, 20, 50]:
    contrib = 1.0 / (K + rank)
    print(f"  Rank {rank:3d}: contribution = {contrib:.6f}")

RRF formula: score(d) = SUM_i [ 1 / (k + rank_i(d)) ]
Smoothing constant k = 60

Example contribution values (k=60):
  Rank   1: contribution = 0.016393
  Rank   2: contribution = 0.016129
  Rank   5: contribution = 0.015385
  Rank  10: contribution = 0.014286
  Rank  20: contribution = 0.012500
  Rank  50: contribution = 0.009091


In [5]:
# ## Part 4: Computing RRF Scores
# We compute the RRF score for every document by combining the BM25
# and dense rankings.

ranked_lists = [bm25_ranked, dense_ranked]

# Compute RRF scores for all documents.
rrf_scores = {}
all_doc_indices = set(bm25_ranked).union(set(dense_ranked))

for doc_idx in all_doc_indices:
    rrf_scores[doc_idx] = rrf_score(doc_idx, ranked_lists, k=K)

# Sort by RRF score descending.
rrf_ranked = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)

print("RRF merged top 10:")
for rank, idx in enumerate(rrf_ranked[:10], 1):
    preview = paragraphs[idx][:60].replace("\n", " ")
    bm25_pos = list(bm25_ranked).index(idx) + 1
    dense_pos = list(dense_ranked).index(idx) + 1
    print(
        f"  #{rank} (rrf={rrf_scores[idx]:.6f}) "
        f"[bm25=#{bm25_pos}, dense=#{dense_pos}]: {preview}..."
    )

RRF merged top 10:
  #1 (rrf=0.032522) [bm25=#1, dense=#2]: “Let’s go on with the game,” the Queen said to Alice; and Al...
  #2 (rrf=0.032018) [bm25=#4, dense=#1]: “Call it what you like,” said the Cat. “Do you play croquet ...
  #3 (rrf=0.031746) [bm25=#3, dense=#3]: “Get to your places!” shouted the Queen in a voice of thunde...
  #4 (rrf=0.031054) [bm25=#2, dense=#7]: The Fish-Footman began by producing from under his arm a gre...
  #5 (rrf=0.030536) [bm25=#6, dense=#5]: “Here! you may nurse it a bit, if you like!” the Duchess sai...
  #6 (rrf=0.029469) [bm25=#5, dense=#11]: The other guests had taken advantage of the Queen’s absence,...
  #7 (rrf=0.028992) [bm25=#8, dense=#10]: CHAPTER I.     Down the Rabbit-Hole  CHAPTER II.    The Pool...
  #8 (rrf=0.028577) [bm25=#11, dense=#9]: “Not at all,” said Alice: “she’s so extremely—” Just then sh...
  #9 (rrf=0.027746) [bm25=#7, dense=#18]: Alice thought she might as well go back, and see how the gam...
  #10 (rrf=0.027673) [bm25=#23, 

In [6]:
# ## Part 5: Why RRF Works -- Document Diversity
# RRF naturally boosts documents that appear in BOTH lists.
# A document at rank 5 in BM25 and rank 8 in dense gets more
# total score than a document at rank 1 in only one list.

print("RRF diversity example:")
print()

# Find a document in the top 10 of both lists.
overlap = [d for d in rrf_ranked[:10] if d in bm25_ranked[:10] and d in dense_ranked[:10]]
exclusive_bm25 = [d for d in rrf_ranked[:10] if d in bm25_ranked[:10] and d not in dense_ranked[:10]]
exclusive_dense = [d for d in rrf_ranked[:10] if d not in bm25_ranked[:10] and d in dense_ranked[:10]]

print(f"Documents in top 10 of BOTH lists: {len(overlap)}")
print(f"Documents ONLY in BM25 top 10:    {len(exclusive_bm25)}")
print(f"Documents ONLY in dense top 10:   {len(exclusive_dense)}")
print()
print("RRF rewards documents found by both systems, giving more robust results.")

RRF diversity example:

Documents in top 10 of BOTH lists: 6
Documents ONLY in BM25 top 10:    2
Documents ONLY in dense top 10:   2

RRF rewards documents found by both systems, giving more robust results.


In [7]:
# ## Part 6: The Effect of k on Fusion
# Smaller k means rank differences matter more.
# Larger k means all ranks contribute more equally.

print("Effect of k on merged ranking:")
print()
for k_val in [1, 10, 30, 60, 100]:
    scores_k = {}
    for doc_idx in all_doc_indices:
        scores_k[doc_idx] = rrf_score(doc_idx, ranked_lists, k=k_val)
    top3 = sorted(scores_k.keys(), key=lambda x: scores_k[x], reverse=True)[:3]
    top3_ids = [f"doc_{d}" for d in top3]
    print(f"  k={k_val:3d}: top 3 = {top3_ids}")

Effect of k on merged ranking:

  k=  1: top 3 = ['doc_365', 'doc_213', 'doc_315']
  k= 10: top 3 = ['doc_365', 'doc_213', 'doc_315']
  k= 30: top 3 = ['doc_365', 'doc_213', 'doc_315']
  k= 60: top 3 = ['doc_365', 'doc_213', 'doc_315']
  k=100: top 3 = ['doc_365', 'doc_213', 'doc_315']


In [8]:
# ## Part 7: Building a Reusable RRF Function
# Let us package this into a clean, reusable function.

def reciprocal_rank_fusion(rankings, k=60):
    """Combine multiple ranked lists using Reciprocal Rank Fusion.

    Args:
        rankings: list of lists, each list is document indices in rank order.
        k: smoothing constant (default 60, per original paper).

    Returns:
        List of (doc_index, rrf_score) sorted by score descending.
    """
    scores = {}
    for ranked_list in rankings:
        for rank, doc_idx in enumerate(ranked_list, 1):
            if doc_idx not in scores:
                scores[doc_idx] = 0.0
            scores[doc_idx] += 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# Test the function.
fused = reciprocal_rank_fusion([bm25_ranked[:20], dense_ranked[:20]], k=K)
print("RRF function output (top 5):")
for rank, (doc_idx, score) in enumerate(fused[:5], 1):
    preview = paragraphs[doc_idx][:60].replace("\n", " ")
    print(f"  #{rank}: score={score:.6f} -- {preview}...")

RRF function output (top 5):
  #1: score=0.032522 -- “Let’s go on with the game,” the Queen said to Alice; and Al...
  #2: score=0.032018 -- “Call it what you like,” said the Cat. “Do you play croquet ...
  #3: score=0.031746 -- “Get to your places!” shouted the Queen in a voice of thunde...
  #4: score=0.031054 -- The Fish-Footman began by producing from under his arm a gre...
  #5: score=0.030536 -- “Here! you may nurse it a bit, if you like!” the Duchess sai...


In [9]:
# ## Part 8: Key Takeaways
# RRF is a simple yet powerful rank fusion method:
# - It requires no training or calibration.
# - It works with any number of ranked lists.
# - It naturally balances contributions from different systems.
# - The k parameter controls how much rank position matters.
#
# The next notebook puts everything together: a full RAG pipeline
# with hybrid retrieval and LLM generation.

print("Summary:")
print("  RRF merges ranked lists using reciprocal rank scoring")
print("  Formula: score(d) = SUM [ 1 / (k + rank) ] across all lists")
print("  k=60 is the standard default from the original paper")
print("  RRF rewards documents appearing in multiple retrieval systems")
print("  It requires no training and works with any number of systems")

Summary:
  RRF merges ranked lists using reciprocal rank scoring
  Formula: score(d) = SUM [ 1 / (k + rank) ] across all lists
  k=60 is the standard default from the original paper
  RRF rewards documents appearing in multiple retrieval systems
  It requires no training and works with any number of systems
